# 06 · Textos: noticias y filings

**Objetivo:** reunir y limpiar las dos fuentes de **texto no estructurado** del proyecto: las noticias de cada empresa y el texto de sus presentaciones a la SEC (filings). Aquí no se calcula ningún sentimiento: solo se prepara el texto que usarán los modelos de los notebooks 07 y 08.

**Entradas:**
- `data/raw/fnspid/All_external.csv`: dataset FNSPID (se descarga en la sección 1 si no existe)
- `data/cache/universo_recortado.parquet` (notebook 02): las 1.000 empresas
- `data/cache/senales.parquet` (notebook 05): para saber qué empresas tienen señal
- `data/cache/filings_sec.parquet` y `empresas_sec.parquet` (notebook 03): qué filings hay y quién presenta 10-K
- Claves `NEWSAPI_KEY` y `SEC_EDGAR_USER_AGENT` en el `.env`

**Salidas:**
- `data/cache/noticias_fnspid.parquet`: titulares de nuestras empresas, oct 2016 – jun 2020
- `data/cache/noticias_newsapi.parquet`: noticias del último mes de las empresas con señal, se acumulan en cada ejecución
- `data/cache/filings_texto.parquet`: apartado de riesgos y MD&A de los 10-K y 10-Q (caché por documento en `data/cache/filings_texto/`)

In [1]:
import json
import re
import time
import warnings

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from dotenv import load_dotenv

from funciones import CARPETA_CACHE, SEC_PAUSA_ENTRE_PETICIONES, obtener_user_agent_sec

# Algunos filings son XHTML: BeautifulSoup avisa, pero el texto se extrae bien
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

load_dotenv();

## 1. ¿Qué noticias históricas tenemos de nuestras empresas?

Las APIs gratuitas de noticias solo dan las últimas semanas. Para tener histórico se usa **FNSPID** (Dong, Fan y Peng, 2024, [arXiv 2402.06698](https://arxiv.org/abs/2402.06698)), un dataset público de noticias financieras en el que cada titular lleva el **ticker** de la empresa. Licencia CC BY-NC 4.0: uso académico citando el artículo.

### 1.1 Descarga

El fichero ocupa 5,7 GB. Solo se descarga si no está ya en disco.

In [2]:
URL_FNSPID = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/All_external.csv"
RUTA_FNSPID = CARPETA_CACHE.parent / "raw" / "fnspid" / "All_external.csv"
TAMANO_FNSPID = 5_731_397_037   # bytes, comprobado con el servidor el 19 oct 2026

if not RUTA_FNSPID.exists():
    print("Descargando FNSPID (5,7 GB)...")
    RUTA_FNSPID.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(URL_FNSPID, stream=True, timeout=60) as respuesta:
        respuesta.raise_for_status()
        with open(RUTA_FNSPID, "wb") as fichero:
            for bloque in respuesta.iter_content(chunk_size=8 * 1024 * 1024):
                fichero.write(bloque)

tamano = RUTA_FNSPID.stat().st_size
if tamano != TAMANO_FNSPID:
    raise RuntimeError(f"FNSPID incompleto: {tamano} bytes en disco, se esperaban {TAMANO_FNSPID}. Borra el fichero y vuelve a ejecutar.")
print(f"FNSPID en disco: {tamano / 1e9:.2f} GB")

FNSPID en disco: 5.73 GB


### 1.2 Cómo es el fichero por dentro

Antes de procesarlo, las primeras filas tal como vienen:

In [3]:
pd.read_csv(RUTA_FNSPID, nrows=5)

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,2020-06-05 06:30:54 UTC,Stocks That Hit 52-Week Highs On Friday,A,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-03 06:45:20 UTC,Stocks That Hit 52-Week Highs On Wednesday,A,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-05-26 00:30:07 UTC,71 Biggest Movers From Friday,A,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-05-22 08:45:06 UTC,46 Stocks Moving In Friday's Mid-Day Session,A,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-05-22 07:38:59 UTC,B of A Securities Maintains Neutral on Agilent...,A,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,NaN,NaN,NaN,NaN,NaN,NaN


Nos interesan cuatro columnas: `Date` (fecha), `Article_title` (titular), `Stock_symbol` (ticker) y `Publisher` (medio). La columna `Article` (texto completo) está vacía en casi todas las filas de nuestras empresas, así que no se usa.

### 1.3 Quedarse con nuestras empresas desde octubre de 2016

Con 13 millones de filas el fichero no cabe cómodo en memoria, así que se lee **por trozos** de 500.000 filas y de cada trozo se guardan solo las filas de nuestras 1.000 empresas desde octubre de 2016 (el inicio de nuestros precios).

FNSPID escribe las clases de acciones con punto (`BRK.B`) y nosotros con guion (`BRK-B`): se unifica antes de filtrar. El resultado se guarda en caché para no volver a leer los 5,7 GB.

In [4]:
RUTA_NOTICIAS_FNSPID = CARPETA_CACHE / "noticias_fnspid.parquet"
universo = pd.read_parquet(CARPETA_CACHE / "universo_recortado.parquet")
nuestros_tickers = set(universo["ticker"])

if RUTA_NOTICIAS_FNSPID.exists():
    filtrado = None
    print("Ya filtrado en una ejecución anterior: se lee de la caché en la sección 1.5")
else:
    trozos = []
    columnas = ["Date", "Article_title", "Stock_symbol", "Publisher"]
    for trozo in pd.read_csv(RUTA_FNSPID, usecols=columnas, dtype=str, chunksize=500_000):
        trozo["ticker"] = trozo["Stock_symbol"].str.replace(".", "-", regex=False)
        trozo["fecha"] = pd.to_datetime(trozo["Date"].str[:10])
        es_nuestro = trozo["ticker"].isin(nuestros_tickers) & (trozo["fecha"] >= "2016-10-01")
        trozos.append(trozo[es_nuestro])
    filtrado = pd.concat(trozos, ignore_index=True)
    filtrado = filtrado.rename(columns={"Article_title": "titular", "Publisher": "medio"})[["fecha", "ticker", "titular", "medio"]]
    print(f"Titulares de nuestras empresas desde oct 2016: {len(filtrado):,}")

Titulares de nuestras empresas desde oct 2016: 486,260


### 1.4 ¿Cuánto ruido traen?

Al mirar los titulares aparecen dos problemas:

1. **Duplicados:** el mismo titular repetido para la misma empresa.
2. **Titulares que no hablan de la empresa:** FNSPID asigna a cada empresa todo lo que aparece en su página de noticias, y muchos titulares son de otras empresas o del mercado en general. Ejemplo real: *"Altria: Income And Growth Together In A Defensive Business"* asignado a Texas Instruments.

Para el segundo se marca si el titular **menciona a la empresa**: su ticker como palabra suelta, o la palabra clave de su nombre. La palabra clave es la primera palabra del nombre (`HORMEL FOODS CORP` → `HORMEL`), o las dos primeras si la primera es demasiado común (`AMERICAN AIRLINES`).

In [5]:
PALABRAS_DEMASIADO_COMUNES = {
    "AMERICAN", "GENERAL", "FIRST", "UNITED", "NATIONAL", "INTERNATIONAL", "TEXAS", "WESTERN", "EASTERN",
    "SOUTHERN", "NORTHERN", "GLOBAL", "THE", "NEW", "CENTRAL", "PACIFIC", "ATLANTIC", "ADVANCED", "APPLIED",
    "DIGITAL", "PUBLIC", "ROYAL", "STANDARD", "UNIVERSAL", "CAPITAL", "BANK", "ENERGY", "HEALTH", "CHINA",
    "CANADIAN", "PACKAGING", "ALLIED", "CONSOLIDATED", "WASTE",
}


def palabra_clave_empresa(nombre: str) -> str:
    """Primera palabra del nombre en mayúsculas, o las dos primeras si la primera es demasiado común."""
    palabras = re.sub(r"[^A-Za-z0-9& ]", " ", nombre).upper().split()
    if palabras[0] in PALABRAS_DEMASIADO_COMUNES and len(palabras) > 1:
        return palabras[0] + " " + palabras[1]
    return palabras[0]


def menciona_empresa(titular: str, ticker: str, palabra_clave: str) -> bool:
    """True si el titular contiene el ticker o la palabra clave como palabras sueltas."""
    titular = titular.upper()
    if re.search(r"\b" + re.escape(ticker) + r"\b", titular):
        return True
    # Una palabra clave de 1-2 letras (p. ej. "X") aparecería en cualquier titular: solo vale el ticker
    return len(palabra_clave) >= 3 and re.search(r"\b" + re.escape(palabra_clave) + r"\b", titular) is not None


universo["palabra_clave"] = universo["name"].map(palabra_clave_empresa)
universo[["ticker", "name", "palabra_clave"]].sample(8, random_state=1)

,ticker,name,palabra_clave
507,LH,LABCORP HOLDINGS INC.,LABCORP
818,FUTU,Futu Holdings Ltd,FUTU
452,QXO,"QXO, Inc.",QXO
368,CRS,CARPENTER TECHNOLOGY CORP,CARPENTER
242,TDG,TransDigm Group INC,TRANSDIGM
929,MTN,VAIL RESORTS INC,VAIL
262,EOG,EOG RESOURCES INC,EOG
810,BEN,FRANKLIN TEMPLETON INC,FRANKLIN


### 1.5 Limpiar y guardar

Se quitan los duplicados (misma empresa y mismo titular) y se añade la columna `sobre_la_empresa`. **No se borran** los titulares que no hablan de la empresa: no sirven para el sentimiento de esa empresa, pero sí como texto financiero para preentrenar los modelos del notebook 07.

In [6]:
if filtrado is not None:
    noticias = filtrado.dropna(subset=["titular"]).drop_duplicates(subset=["ticker", "titular"])
    noticias = noticias.merge(universo[["ticker", "palabra_clave"]], on="ticker")
    noticias["sobre_la_empresa"] = [
        menciona_empresa(titular, ticker, palabra)
        for titular, ticker, palabra in zip(noticias["titular"], noticias["ticker"], noticias["palabra_clave"])
    ]
    noticias = noticias.drop(columns="palabra_clave").sort_values(["ticker", "fecha"]).reset_index(drop=True)
    noticias.to_parquet(RUTA_NOTICIAS_FNSPID, index=False)
    print(f"Duplicados eliminados: {len(filtrado) - len(noticias):,}")

noticias = pd.read_parquet(RUTA_NOTICIAS_FNSPID)
print(f"Titulares guardados: {len(noticias):,} · sobre la propia empresa: {noticias['sobre_la_empresa'].mean():.0%}")

Duplicados eliminados: 21,277
Titulares guardados: 464,983 · sobre la propia empresa: 42%


Ejemplos de cada tipo, para comprobar a ojo que la marca tiene sentido:

In [7]:
pd.set_option("display.max_colwidth", 110)
display(noticias[noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])
display(noticias[~noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])

,fecha,ticker,titular
245912,2016-11-22,KKR,KKR launches $4.5B offer for Calsonic Kansei
199301,2017-03-06,HAL,Halliburton Announces CFO Mark McCollum Leaving Co.; General Counsel Robb Voyles Named Interim CFO
401693,2019-10-24,SYK,What's in the Offing for Stryker's (SYK) Earnings in Q3?
368479,2019-05-13,RGLD,"RBC Capital Downgrades Royal Gold, Inc. - Common Stock to Sector Perform"
412118,2020-05-15,TMO,"Blue Pool Management Ltd. Buys Adobe Inc, Facebook Inc, Thermo Fisher Scientific Inc, Sells ..."
218076,2020-03-17,ICLR,"SunTrust Robinson Humphrey Maintains Hold on Icon, Lowers Price Target to $131"


,fecha,ticker,titular
364904,2017-08-01,REGN,Stocks Rally: 2 Leaders Break Out On Earnings
172425,2019-12-14,FE,Stocks To Watch: Investors Dig Into Trade War Halt
156143,2019-08-23,EOG,Shares of several oil & gas companies are trading lower on escalated trade tensions after China announced ...
116267,2018-01-15,CSX,Sales Growth + Dividend Growth = Danger!
399521,2017-01-04,SUI,Can You Afford To Retire? Current And Future Portfolio Income
397867,2019-01-07,STZ,New Age Beverages: Morinda Acquisition To Lead To Higher Revenues


### 1.6 Cobertura: qué empresas y qué años

Solo cuentan los titulares sobre la propia empresa, que son los que se usarán para su sentimiento.

In [8]:
sobre_empresa = noticias[noticias["sobre_la_empresa"]]
por_empresa = sobre_empresa.groupby("ticker").size()

print(f"Periodo: {noticias['fecha'].min().date()} → {noticias['fecha'].max().date()}")
print(f"Empresas con algún titular propio: {len(por_empresa)} de {len(universo)}")
print(f"Titulares propios por empresa: mediana {por_empresa.median():.0f} · 10 % de las empresas con menos de {por_empresa.quantile(0.1):.0f}")

universo["con_noticias"] = universo["ticker"].isin(por_empresa.index)
print(universo.groupby("exchange")["con_noticias"].mean().mul(100).round(0).rename("% de empresas con noticias").to_string())

Periodo: 2016-10-01 → 2020-06-11
Empresas con algún titular propio: 633 de 1000
Titulares propios por empresa: mediana 265 · 10 % de las empresas con menos de 75
exchange
NYSE      69.0
Nasdaq    54.0


In [9]:
por_mes = sobre_empresa.groupby(pd.Grouper(key="fecha", freq="ME")).size().reset_index(name="titulares")
grafico = px.bar(por_mes, x="fecha", y="titulares",
                 title="Titulares sobre la propia empresa, por mes (FNSPID, nuestras 1.000 empresas)")
grafico.show(renderer="notebook_connected")

**Conclusión:** FNSPID da noticias de unas dos terceras partes de nuestras empresas entre octubre de 2016 y junio de 2020. Menos de la mitad de los titulares tratan de la propia empresa; esos son los que medirán su sentimiento, y el resto se aprovecha como corpus. De las 367 empresas sin noticias, 119 empezaron a cotizar después de junio de 2020; las demás (incluidas algunas grandes, como Cardinal Health o Ferrari) simplemente no están en FNSPID. Ninguna de ellas podrá participar en la comparación noticias/filings, y esa limitación se documenta.

## 2. ¿Qué se dice ahora de las empresas con señal?

FNSPID termina en junio de 2020. Para el screener del día y para el test que se etiqueta a mano hacen falta noticias **recientes**. Se usa **NewsAPI**, con los límites del plan gratuito medidos el 19 oct 2026:

| Límite | Valor |
|---|---|
| Peticiones | 100 al día |
| Antigüedad máxima | un mes |
| Resultados por consulta | 100 como máximo |
| Retraso | ~1 día |

Con 100 peticiones no se pueden pedir las 1.000 empresas cada día, así que se piden solo las que tienen **señal el último día** de nuestros precios, en grupos de 3 por consulta.

Dos decisiones tomadas al ver los datos reales:

- **Solo medios financieros** (Yahoo Finance, MarketWatch, Benzinga, CNBC, PR Newswire…). Sin este filtro, buscar "Garmin" devuelve sorteos de relojes y paquetes de software.
- **Solo en el titular** (`searchIn=title`), igual que la marca `sobre_la_empresa` de FNSPID.

In [10]:
import os

URL_NEWSAPI = "https://newsapi.org/v2/everything"
MEDIOS_FINANCIEROS = [
    "finance.yahoo.com", "marketwatch.com", "fool.com", "seekingalpha.com", "benzinga.com", "cnbc.com",
    "investors.com", "zacks.com", "barrons.com", "wsj.com", "bloomberg.com", "reuters.com", "forbes.com",
    "businesswire.com", "prnewswire.com", "globenewswire.com", "biztoc.com", "thestreet.com", "investing.com",
    "nasdaq.com", "businessinsider.com", "ft.com", "axios.com", "gurufocus.com", "insidermonkey.com", "247wallst.com",
]


def buscar_titulares_newsapi(palabras: list[str], desde: pd.Timestamp) -> list[dict]:
    """Busca en los titulares de medios financieros cualquiera de las palabras, desde la fecha dada.

    Devuelve la lista de artículos tal como la da NewsAPI. Lanza RuntimeError si falta la
    clave o si la API responde con un error (por ejemplo, el límite diario agotado).
    """
    clave = os.getenv("NEWSAPI_KEY", "").strip()
    if not clave:
        raise RuntimeError("Falta NEWSAPI_KEY en el .env")
    parametros = {
        "q": " OR ".join(f'"{palabra}"' for palabra in palabras),
        "searchIn": "title",
        "domains": ",".join(MEDIOS_FINANCIEROS),
        "language": "en",
        "from": desde.strftime("%Y-%m-%d"),
        "sortBy": "publishedAt",
        "pageSize": 100,
    }
    respuesta = requests.get(URL_NEWSAPI, params=parametros, headers={"X-Api-Key": clave}, timeout=30)
    datos = respuesta.json()
    if respuesta.status_code != 200:
        raise RuntimeError(f"NewsAPI respondió {respuesta.status_code}: {datos.get('code')} - {datos.get('message')}")
    return datos["articles"]

Así llega una respuesta real (una consulta de prueba con Garmin): campos y un ejemplo. Se guarda en caché para no gastar una petición en cada ejecución.

In [11]:
import json

RUTA_PRUEBA_NEWSAPI = CARPETA_CACHE / "newsapi_respuesta_ejemplo.json"
if RUTA_PRUEBA_NEWSAPI.exists():
    prueba = json.loads(RUTA_PRUEBA_NEWSAPI.read_text())
else:
    prueba = buscar_titulares_newsapi(["Garmin"], desde=pd.Timestamp.today() - pd.Timedelta(days=29))
    RUTA_PRUEBA_NEWSAPI.write_text(json.dumps(prueba, indent=1))
print(f"Artículos: {len(prueba)} · campos: {list(prueba[0])}")
prueba[0]

Artículos: 9 · campos: ['source', 'author', 'title', 'description', 'url', 'urlToImage', 'publishedAt', 'content']


{'source': {'id': None, 'name': 'PRNewswire'},
 'author': None,
 'title': 'Garmin selects Ventura, California as newest Garmin Marathon Series location',
 'description': 'Event will showcase breathtaking mountain and ocean scenery in August 2027 OLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a fourth location in Ventura, California, on Sunday, Aug. 22, …',
 'url': 'https://www.prnewswire.com/news-releases/garmin-selects-ventura-california-as-newest-garmin-marathon-series-location-302908287.html',
 'urlToImage': 'https://mmx.prnewswire.com/media/MS1128164/Garmin-Marathon-Series.jpg?id=OA3016159&p=facebook',
 'publishedAt': '2026-10-14T14:02:00Z',
 'content': 'Event will showcase breathtaking mountain and ocean scenery in August 2027\r\nOLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a… [+5337 chars]'}

### 2.1 Descargar y acumular

Las empresas a consultar son las que tienen señal el último día de `senales.parquet`. Cada titular se asigna a las empresas de su grupo que **menciona** (la misma regla que en FNSPID).

Como NewsAPI solo guarda un mes, cada ejecución **se suma** a lo ya guardado (sin repetir artículos). Si ya se descargó hoy, no se vuelve a pedir nada: así re-ejecutar el notebook no gasta el cupo diario.

In [12]:
RUTA_NOTICIAS_NEWSAPI = CARPETA_CACHE / "noticias_newsapi.parquet"
EMPRESAS_POR_CONSULTA = 3
hoy = pd.Timestamp.today().normalize()

senales = pd.read_parquet(CARPETA_CACHE / "senales.parquet", columns=["fecha", "ticker", "senal"])
ultimo_dia = senales["fecha"].max()
con_senal = senales[(senales["fecha"] == ultimo_dia) & senales["senal"].isin(["compra", "venta"])]
empresas = universo[universo["ticker"].isin(con_senal["ticker"]) & (universo["palabra_clave"].str.len() >= 3)]
print(f"Empresas con señal el {ultimo_dia.date()}: {len(empresas)} → {-(-len(empresas) // EMPRESAS_POR_CONSULTA)} consultas")

guardadas = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI) if RUTA_NOTICIAS_NEWSAPI.exists() else None

if guardadas is not None and guardadas["descargado"].max() >= hoy:
    print("Ya se descargó hoy: no se hace ninguna petición")
else:
    filas = []
    consultas_llenas = 0
    for inicio in range(0, len(empresas), EMPRESAS_POR_CONSULTA):
        grupo = empresas.iloc[inicio : inicio + EMPRESAS_POR_CONSULTA]
        articulos = buscar_titulares_newsapi(list(grupo["palabra_clave"]), desde=hoy - pd.Timedelta(days=29))
        if len(articulos) == 100:
            consultas_llenas += 1
        for articulo in articulos:
            for empresa in grupo.itertuples():
                if menciona_empresa(articulo["title"], empresa.ticker, empresa.palabra_clave):
                    filas.append({
                        "fecha": pd.Timestamp(articulo["publishedAt"][:10]),
                        "ticker": empresa.ticker,
                        "titular": articulo["title"],
                        "resumen": articulo["description"],
                        "medio": articulo["source"]["name"],
                        "url": articulo["url"],
                        "descargado": hoy,
                    })
    nuevas = pd.DataFrame(filas)
    print(f"Artículos nuevos asignados: {len(nuevas)} · consultas que llegaron al tope de 100: {consultas_llenas}")
    if guardadas is not None:
        nuevas = pd.concat([guardadas, nuevas], ignore_index=True)
    guardadas = nuevas.drop_duplicates(subset=["ticker", "url"])
    guardadas.to_parquet(RUTA_NOTICIAS_NEWSAPI, index=False)

noticias_recientes = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI)
print(f"Noticias recientes guardadas: {len(noticias_recientes)} · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · de {noticias_recientes['fecha'].min().date()} a {noticias_recientes['fecha'].max().date()}")

Empresas con señal el 2026-10-09: 99 → 33 consultas
Ya se descargó hoy: no se hace ninguna petición
Noticias recientes guardadas: 1494 · empresas: 89 · de 2026-09-20 a 2026-10-19


In [13]:
display(noticias_recientes.sample(8, random_state=1)[["fecha", "ticker", "medio", "titular"]])
noticias_recientes["medio"].value_counts().head(8)

,fecha,ticker,medio,titular
91,2026-10-12,PLTR,Biztoc.com,Palantir Stock Gets Fresh Wall Street Vote of Confidence
75,2026-10-13,AMZN,Biztoc.com,"Amazon News, Politico and Warner Bros. Discovery are finalists for the 2026 Digiday Awards finalists"
1209,2026-10-01,LUNR,Biztoc.com,Is Wall Street Dead Wrong About Intuitive Surgical? Here's My Honest Answer
330,2026-10-05,SHOP,Biztoc.com,Why Shopify Stock Rallied Today
959,2026-10-12,SNAP,CNBC,A massive trade just happened in surging shares of Snap
742,2026-10-15,MSCI,Biztoc.com,"MSCI finalizes index rules that put Strategy, crypto treasury firms at risk of exclusion"
893,2026-09-20,JOBY,Biztoc.com,Joby Aviation Stock Is Down 54% This Year. Here's Why I'd Buy It Before 2027
1194,2026-09-22,FSLY,24/7 Wall St.,Fastly Tumbles 7% as Investor Day Meets Profit Taking; Cloudflare and Akamai Edge Higher


medio
Biztoc.com                   1134
PRNewswire                    283
CNBC                           24
24/7 Wall St.                  24
Business Insider               20
Investor's Business Daily       6
TheStreet                       2
Forbes                          1
Name: count, dtype: int64

**Conclusión:** NewsAPI aporta titulares **y resumen** de las empresas con señal, solo del último mes. Casi todo llega a través de Biztoc (un agregador) y PR Newswire (notas de prensa de las propias empresas). La regla de mención falla a veces con nombres que son palabras comunes: a Pinnacle West (`PNW`) se le asigna un titular sobre un "Pinnacle Lifetime Member". Algunas consultas llegan al tope de 100 artículos, así que de las empresas más populares puede faltar alguna noticia. Cada ejecución en un día distinto suma noticias, así que el histórico reciente crece mientras se trabaja en el proyecto.

## 3. ¿Qué dicen las empresas de sí mismas? (filings de la SEC)

La segunda fuente de texto son los informes que las empresas de EEUU presentan por ley a la SEC:

| | **10-K** | **10-Q** |
|---|---|---|
| Qué es | Informe **anual**, con cuentas auditadas | Informe **trimestral** (3 al año) |
| Riesgos | **Item 1A**: todos los riesgos del negocio | **Part II, Item 1A**: solo los cambios desde el último 10-K |
| Análisis de la dirección (MD&A) | **Item 7**: la dirección explica el año | **Part I, Item 2**: lo mismo para el trimestre |

Se extraen esas dos partes, **riesgos** y **MD&A**, porque son el texto en el que la propia empresa habla de su situación. Alcance decidido por el autor el 20 oct 2026:

- **10-K y 10-Q** de las empresas que presentan 10-K. Las ~110 extranjeras (20-F/40-F) quedan fuera: sus informes tienen otra estructura.
- **De octubre de 2016 a junio de 2020** (el periodo con noticias de FNSPID), más el **último 10-K y el último 10-Q** de cada empresa para el screener.
- XOM con los filings de su CIK anterior (notebook 03).

In [14]:
filings = pd.read_parquet(CARPETA_CACHE / "filings_sec.parquet")
empresas_sec = pd.read_parquet(CARPETA_CACHE / "empresas_sec.parquet")
filings = filings.merge(empresas_sec[["cik", "ticker", "formulario_anual"]], on="cik")
filings = filings[filings["formulario"].isin(["10-K", "10-Q"]) & (filings["formulario_anual"] == "10-K")]

en_el_periodo = filings[(filings["fecha_presentacion"] >= "2016-10-01") & (filings["fecha_presentacion"] <= "2020-06-30")]
los_ultimos = filings.sort_values("fecha_presentacion").groupby(["cik", "formulario"]).tail(1)
documentos = pd.concat([en_el_periodo, los_ultimos]).drop_duplicates(subset=["cik", "numero_acceso"]).reset_index(drop=True)

print(f"Documentos a procesar: {len(documentos):,} de {documentos['cik'].nunique()} empresas")
documentos["formulario"].value_counts()

Documentos a procesar: 12,523 de 883 empresas


formulario
10-Q    8830
10-K    3693
Name: count, dtype: int64

### 3.1 Cómo es un filing por dentro

Cada filing es una página HTML de varios MB. Se descarga con el User-Agent que exige la SEC, se le quitan las etiquetas HTML y se juntan los espacios. Así queda el texto plano.

Ejemplo real: el 10-K de Apple de 2019. Buscamos dónde aparece la cabecera "Item 1A":

In [15]:
def descargar_documento_sec(url: str, intentos: int = 3) -> bytes:
    """Descarga un documento de EDGAR con el User-Agent obligatorio y espera entre peticiones.

    Reintenta si la SEC responde 429 (demasiadas peticiones) o un error 5xx. Lanza RuntimeError
    si tras los intentos no lo consigue.
    """
    for intento in range(1, intentos + 1):
        try:
            respuesta = requests.get(url, headers={"User-Agent": obtener_user_agent_sec()}, timeout=60)
        except requests.exceptions.RequestException as error:
            motivo = str(error)
        else:
            time.sleep(SEC_PAUSA_ENTRE_PETICIONES)
            if respuesta.status_code == 200:
                return respuesta.content
            motivo = f"HTTP {respuesta.status_code}"
            if respuesta.status_code not in (429, 500, 502, 503, 504):
                break
        time.sleep(2 * intento)
    raise RuntimeError(f"No se pudo descargar {url}: {motivo}")


def html_a_texto(contenido: bytes) -> str:
    """Texto plano de un documento HTML, con los espacios juntados en uno."""
    texto = BeautifulSoup(contenido, "lxml").get_text(" ").replace("\xa0", " ")
    return re.sub(r"\s+", " ", texto)


def url_del_documento(filing) -> str:
    """Dirección del documento principal de un filing en EDGAR."""
    carpeta = filing.numero_acceso.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{int(filing.cik_archivo)}/{carpeta}/{filing.documento_principal}"


RUTA_EJEMPLO = CARPETA_CACHE / "filing_ejemplo_aapl_10k_2019.txt"
filing_apple = documentos[(documentos["ticker"] == "AAPL") & (documentos["formulario"] == "10-K")
                          & (documentos["fecha_presentacion"].dt.year == 2019)].iloc[0]
if not RUTA_EJEMPLO.exists():
    RUTA_EJEMPLO.write_text(html_a_texto(descargar_documento_sec(url_del_documento(filing_apple))))
texto_apple = RUTA_EJEMPLO.read_text()

print(f"10-K de Apple ({filing_apple['fecha_presentacion'].date()}): {len(texto_apple):,} caracteres")
for aparicion in re.finditer(r"item\s*1a", texto_apple, flags=re.I):
    print(f"  posición {aparicion.start():>7}: …{texto_apple[aparicion.start() - 20 : aparicion.start() + 60]}…")

10-K de Apple (2019-10-31): 262,791 caracteres
  posición   29125: … Item 1. Business 1 Item 1A. Risk Factors 5 Item 1B. Unresolved Staff Comments 1…
  posición   31362: …iscussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors,” …
  posición   45123: … 2019 Form 10-K | 4 Item 1A. Risk Factors The following discussion of risk facto…
  posición  101630: …rights ” in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.” …
  posición  108153: …iscussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors,” …
  posición  117632: …et forth in Part I, Item 1A of this Form 10-K under the heading “Risk Factors”. …


La cabecera aparece **varias veces**:

1. En el **índice** del principio, seguida del número de página.
2. Como **referencia** dentro del texto: *"…under the heading “Risk Factors” in Item 1A of this Form 10-K"*.
3. Como **cabecera real** de la sección, seguida del texto de los riesgos.

### 3.2 Cómo se localiza cada sección

Para cada sección hay una cabecera de **inicio** y una lista de cabeceras de **final**, en orden de preferencia (en un 10-K, los riesgos acaban en el Item 1B y, si no lo hay, en el Item 2). Las reglas:

- Se **descartan las referencias**: cabeceras precedidas de comillas, *see*, *in*, *under*, *of* o *and*, o seguidas de comillas (también tras un punto, una coma o un guion: *Risk Factors."*, *Risk Factors - "…"*).
- De cada inicio se corta hasta el **primer final** que aparece después. Si ese final no está, se prueba el siguiente de la lista.
- Entre todos los candidatos se queda el **más largo**. El del índice mide unos pocos caracteres; el de la sección real, decenas de miles.
- Si la sección ocupa **más del 75 % del documento**, se descarta: el final no se encontró y el corte ha arrastrado casi todo el informe. En una muestra de 80 documentos, las secciones correctas llegaban como mucho al 68 % y las erróneas estaban entre el 90 % y el 95 %.

Si no se encuentra la sección, queda vacía y se cuenta. Algunas empresas usan un formato propio sin cabeceras "Item", como Cardinal Health en 2026.

In [16]:
SEPARADOR = r"[\s\.\:\-—–]*"   # entre "Item", el número y el título puede haber puntos, guiones o dos puntos
SECCIONES = {
    ("10-K", "riesgos"): (rf"item{SEPARADOR}1a{SEPARADOR}risk\s+factors",
                          [rf"item{SEPARADOR}1b{SEPARADOR}unresolved", rf"item{SEPARADOR}1c{SEPARADOR}cybersecurity",
                           rf"item{SEPARADOR}2{SEPARADOR}(?:description\s+of\s+)?propert"]),
    ("10-K", "mda"): (rf"item{SEPARADOR}7{SEPARADOR}management.{{0,3}}s?\s+discussion",
                      [rf"item{SEPARADOR}7a{SEPARADOR}quantitative", rf"item{SEPARADOR}8{SEPARADOR}financial\s+statements"]),
    ("10-Q", "riesgos"): (rf"item{SEPARADOR}1a{SEPARADOR}risk\s+factors",
                          [rf"item{SEPARADOR}2{SEPARADOR}unregistered", rf"item{SEPARADOR}3{SEPARADOR}defaults",
                           rf"item{SEPARADOR}4{SEPARADOR}mine", rf"item{SEPARADOR}5{SEPARADOR}other\s+information",
                           rf"item{SEPARADOR}6{SEPARADOR}exhibits"]),
    ("10-Q", "mda"): (rf"item{SEPARADOR}2{SEPARADOR}management.{{0,3}}s?\s+discussion",
                      [rf"item{SEPARADOR}3{SEPARADOR}quantitative", rf"item{SEPARADOR}4{SEPARADOR}controls"]),
}
MAXIMO_DEL_DOCUMENTO = 0.75
ANTES_DE_UNA_REFERENCIA = re.compile(r"(?:[“\"‘]|\bin|\bsee|\bunder|\bof|\band)\s*$", flags=re.I)


def es_referencia(texto: str, inicio: re.Match) -> bool:
    """True si la cabecera encontrada es una cita a la sección y no la sección misma."""
    antes = texto[max(0, inicio.start() - 8) : inicio.start()]
    despues = texto[inicio.end() : inicio.end() + 5].lstrip(" .,;:-–—")   # «Risk Factors."» o «Risk Factors - "…"» también son citas
    return bool(ANTES_DE_UNA_REFERENCIA.search(antes)) or despues[:1] in ("”", '"', "’")


def extraer_seccion(texto: str, formulario: str, parte: str) -> str:
    """Texto de una sección ('riesgos' o 'mda') de un 10-K o 10-Q; cadena vacía si no se encuentra."""
    patron_inicio, patrones_final = SECCIONES[(formulario, parte)]
    mejor = ""
    for inicio in re.finditer(patron_inicio, texto, flags=re.I):
        if es_referencia(texto, inicio):
            continue
        for patron_final in patrones_final:
            final = re.compile(patron_final, flags=re.I).search(texto, inicio.end())
            if final:
                if final.start() - inicio.start() > len(mejor):
                    mejor = texto[inicio.start() : final.start()]
                break
    if len(mejor) > MAXIMO_DEL_DOCUMENTO * len(texto):
        return ""
    return mejor


for parte in ["riesgos", "mda"]:
    seccion = extraer_seccion(texto_apple, "10-K", parte)
    print(f"{parte}: {len(seccion):,} caracteres · empieza: «{seccion[:90]}» · acaba: «{seccion[-60:]}»")

riesgos: 54,890 caracteres · empieza: «Item 1A. Risk Factors The following discussion of risk factors contains forward-looking st» · acaba: «d operating results could be materially adversely affected. »
mda: 33,810 caracteres · empieza: «Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operati» · acaba: «at reporting period could be materially adversely affected. »


### 3.3 Descargar y extraer los ~12.500 documentos

Cada documento se descarga, se extrae y se guarda **en su propio fichero** de `data/cache/filings_texto/`. Si la descarga se corta, al volver a ejecutar se sigue por donde iba, y re-ejecutar el notebook no vuelve a pedir nada a la SEC. La primera vez tarda **entre 1,5 y 2 horas** (unos 3,6 MB y medio segundo por documento, dentro del límite de la SEC). Un documento que falla se anota y se cuenta; no detiene el resto.

In [17]:
CARPETA_FILINGS_TEXTO = CARPETA_CACHE / "filings_texto"
CARPETA_FILINGS_TEXTO.mkdir(exist_ok=True)

errores_filings = {}
inicio_descarga = time.time()
pendientes = [f for f in documentos.itertuples() if not (CARPETA_FILINGS_TEXTO / f"{f.numero_acceso}.json").exists()]
print(f"Ya en caché: {len(documentos) - len(pendientes):,} · pendientes: {len(pendientes):,}")

for numero, filing in enumerate(pendientes, start=1):
    try:
        texto = html_a_texto(descargar_documento_sec(url_del_documento(filing)))
    except RuntimeError as error:
        errores_filings[filing.numero_acceso] = str(error)
        continue
    resultado = {
        "caracteres_documento": len(texto),
        "riesgos": extraer_seccion(texto, filing.formulario, "riesgos"),
        "mda": extraer_seccion(texto, filing.formulario, "mda"),
    }
    (CARPETA_FILINGS_TEXTO / f"{filing.numero_acceso}.json").write_text(json.dumps(resultado))
    if numero % 500 == 0:
        print(f"  {numero:,}/{len(pendientes):,} ({(time.time() - inicio_descarga) / 60:.0f} min)")

print(f"Terminado en {(time.time() - inicio_descarga) / 60:.0f} min · errores: {len(errores_filings)}")
errores_filings

Ya en caché: 12,523 · pendientes: 0
Terminado en 0 min · errores: 0


{}

### 3.4 ¿Se han encontrado las secciones?

Se juntan todos los resultados en una tabla, una fila por filing, y se mide en cuántos documentos se ha encontrado cada sección.

In [18]:
filas = []
for filing in documentos.itertuples():
    ruta = CARPETA_FILINGS_TEXTO / f"{filing.numero_acceso}.json"
    if ruta.exists():
        filas.append({"numero_acceso": filing.numero_acceso, **json.loads(ruta.read_text())})
filings_texto = documentos[["cik", "ticker", "formulario", "fecha_presentacion", "fecha_periodo", "numero_acceso"]].merge(
    pd.DataFrame(filas), on="numero_acceso")
filings_texto["caracteres_riesgos"] = filings_texto["riesgos"].str.len()
filings_texto["caracteres_mda"] = filings_texto["mda"].str.len()
filings_texto.to_parquet(CARPETA_CACHE / "filings_texto.parquet", index=False)

filings_texto["tiene_riesgos"] = filings_texto["caracteres_riesgos"] > 0
filings_texto["tiene_mda"] = filings_texto["caracteres_mda"] > 0
resumen = filings_texto.groupby("formulario").agg(
    documentos=("numero_acceso", "size"),
    con_riesgos=("tiene_riesgos", "mean"),
    caracteres_riesgos_mediana=("caracteres_riesgos", "median"),
    con_mda=("tiene_mda", "mean"),
    caracteres_mda_mediana=("caracteres_mda", "median"),
)
resumen[["con_riesgos", "con_mda"]] = (resumen[["con_riesgos", "con_mda"]] * 100).round(0)   # en %
print(f"Filings con texto guardado: {len(filings_texto):,} de {len(documentos):,} · empresas: {filings_texto['cik'].nunique()}")
resumen

Filings con texto guardado: 12,523 de 12,523 · empresas: 883


,documentos,con_riesgos,caracteres_riesgos_mediana,con_mda,caracteres_mda_mediana
formulario,,,,,
10-K,3693,98.0,65382.0,98.0,72996.0
10-Q,8830,89.0,428.0,97.0,49023.0


In [19]:
print("Inicio de algunas secciones de riesgos de 10-Q (suele ser una sola frase):")
for texto in filings_texto.loc[(filings_texto["formulario"] == "10-Q") & (filings_texto["caracteres_riesgos"] > 0), "riesgos"].sample(5, random_state=3):
    print("  ·", texto[:160])

sin_nada = filings_texto[(filings_texto["caracteres_riesgos"] == 0) & (filings_texto["caracteres_mda"] == 0)]
print(f"\nDocumentos sin ninguna de las dos secciones: {len(sin_nada)} · empresas con más casos:")
sin_nada["ticker"].value_counts().head(8)

Inicio de algunas secciones de riesgos de 10-Q (suele ser una sola frase):
  · ITEM 1A. RISK FACTORS There have been no material changes to the risk factors discussed in our Annual Report. 
  · Item 1A. Risk Factors The factors discussed below are cautionary statements that identify important factors and risks that could cause actual results to differ 
  · Item 1A. Risk Factors In addition to the other information set forth in this Form 10-Q, you should carefully consider the factors discussed in Part I, Item 1A. 
  · Item 1A. Risk Factors. Investors should carefully review and consider the information regarding certain factors that could materially affect our business, resul
  · Item 1A. Risk Factors RISK FACTORS A description of the risks and uncertainties associated with our business is set forth below. You should carefully consider t

Documentos sin ninguna de las dos secciones: 158 · empresas con más casos:


ticker
C       17
EIX     17
MOH     15
MS      15
FITB    12
ETR     11
EMR     11
STT     10
Name: count, dtype: int64

**Conclusión:** cuando la empresa usa las cabeceras estándar "Item 1A" e "Item 7", la extracción encuentra las dos secciones casi siempre. En los 10-Q, el apartado de riesgos suele limitarse a decir que no hay cambios, así que el texto con contenido de los 10-Q es sobre todo el MD&A. Los documentos sin secciones son empresas con un formato propio; quedan sin sentimiento de filings, y su número se documenta.

## Comprobaciones

In [20]:
# Solo nuestras empresas y solo el periodo pedido
assert noticias["ticker"].isin(nuestros_tickers).all()
assert noticias["fecha"].min() >= pd.Timestamp("2016-10-01")

# Sin titulares vacíos ni duplicados por empresa
assert noticias["titular"].notna().all()
assert not noticias.duplicated(subset=["ticker", "titular"]).any()

# La regla de mención, con casos escritos a mano
assert palabra_clave_empresa("HORMEL FOODS CORP /DE/") == "HORMEL"
assert palabra_clave_empresa("AMERICAN AIRLINES GROUP INC.") == "AMERICAN AIRLINES"
assert menciona_empresa("FedEx (FDX) Stock Pops On Q3 Beat", "FDX", "FEDEX")
assert menciona_empresa("Garmin Reports EASA Approval", "GRMN", "GARMIN")
assert not menciona_empresa("Altria: Income And Growth Together", "TXN", "TEXAS INSTRUMENTS")
assert not menciona_empresa("Next steps for X-ray makers", "XE", "X")   # palabra clave demasiado corta
print("Noticias FNSPID: OK")

Noticias FNSPID: OK


In [21]:
# Noticias recientes: sin duplicados, todas mencionan a su empresa y dentro del mes que permite NewsAPI
assert not noticias_recientes.duplicated(subset=["ticker", "url"]).any()
claves = universo.set_index("ticker")["palabra_clave"]
assert all(menciona_empresa(t, k, claves[k]) for t, k in zip(noticias_recientes["titular"], noticias_recientes["ticker"]))
assert (noticias_recientes["descargado"] - noticias_recientes["fecha"] <= pd.Timedelta(days=31)).all()
print("Noticias NewsAPI: OK")

Noticias NewsAPI: OK


In [22]:
# Filings: solo empresas de 10-K, solo 10-K/10-Q y la sección nunca es casi todo el documento
assert set(filings_texto["formulario"]) == {"10-K", "10-Q"}
assert set(filings_texto["cik"]) <= set(empresas_sec.loc[empresas_sec["formulario_anual"] == "10-K", "cik"])
assert (filings_texto["caracteres_riesgos"] <= MAXIMO_DEL_DOCUMENTO * filings_texto["caracteres_documento"]).all()
assert (filings_texto["caracteres_mda"] <= MAXIMO_DEL_DOCUMENTO * filings_texto["caracteres_documento"]).all()

# Casi todos los documentos procesados, y la gran mayoría de 10-K con sus dos secciones
assert len(filings_texto) >= 0.98 * len(documentos)
diez_k = filings_texto[filings_texto["formulario"] == "10-K"]
assert (diez_k["caracteres_riesgos"] > 0).mean() > 0.9 and (diez_k["caracteres_mda"] > 0).mean() > 0.9

# Las reglas de extracción con un caso escrito a mano
caso = 'See Item 1A "Risk Factors". Contents Item 1A. Risk Factors 9 Item 1B. Unresolved 22 ' \
       'Item 1A. Risk Factors We face risks, as detailed in Item 2. Properties. More risks. Item 1B. Unresolved Staff Comments None.'
assert extraer_seccion(caso, "10-K", "riesgos").startswith("Item 1A. Risk Factors We face risks")
assert extraer_seccion(caso, "10-K", "riesgos").endswith("More risks. ")
caso_con_punto = 'described in Part I, Item 1A. Risk Factors." This discussion is about results. Item 1B. Unresolved'
assert extraer_seccion(caso_con_punto, "10-K", "riesgos") == ""

# Ninguna sección guardada empieza en una cita a sí misma
empieza_en_cita = filings_texto["riesgos"].str[:90].str.contains(r"^item\W*1a\W*risk\s+factors\W{0,3}[”\"]", case=False, regex=True)
assert not empieza_en_cita.any(), f"{empieza_en_cita.sum()} secciones de riesgos empiezan en una cita"

# XOM tiene texto de sus 10-K del periodo, bajo el CIK antiguo
assert ((filings_texto["ticker"] == "XOM") & (filings_texto["formulario"] == "10-K")
        & (filings_texto["fecha_presentacion"] < "2020-07-01") & (filings_texto["caracteres_riesgos"] > 0)).any()
print("Filings: OK")

Filings: OK
